In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)
# df = df.drop(columns="Unnamed: 0", axis=1)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
df['Delivery_Time'].hist()

In [ ]:
# Task 1: Write your code here:
df.drop('Order_ID', axis=1, inplace=True)

In [ ]:
# Task 2: Write your code here:
df.isna().sum()

In [ ]:
# for now impute categorical_cols null values with mode and numerical with mean
categorical_cols = df.select_dtypes('object').columns
numerical_cols = df.drop('Delivery_Time', axis=1).select_dtypes('number').columns

df_clean = df.copy()
for col in categorical_cols:
  df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

for col in numerical_cols:
  df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

# drop data with missing target
df_clean = df_clean.dropna(subset='Delivery_Time')

print(f"Shape before cleaning {df.shape}")
print(f"Shape after cleaning {df_clean.shape}")
print(df_clean.isna().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
df_clean.shape

In [ ]:
categorical_cols

In [ ]:
df_clean.isna().sum()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
onehot_encoded = pd.DataFrame(onehot_encoder.fit_transform(df_clean[categorical_cols]), columns=onehot_encoder.get_feature_names_out(df_clean[categorical_cols].columns))

In [ ]:
onehot_encoded.shape

In [ ]:
df_clean = pd.concat([onehot_encoded, df_clean[numerical_cols], df_clean['Delivery_Time']], axis=1, join='inner')

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop('Delivery_Time', axis=1)
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=1,
    random_state=42
)

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
model_losses = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

  # Validate
  y_pred = model.predict(X_test)

  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)

  # Store results
  model_losses.append(mae)
  print(f"Mean Absolute Errors: {mae}")

print(f"Average MAE: {np.mean(mae)}")

In [ ]:
# Task 1: Write your code here:
feature_importance = model.feature_importances_
plt.barh(X.columns, feature_importance)
plt.show()

In [ ]:
# Task 2: Write your code here:
y_pred = model.predict(X)
pd.Series(y_pred).hist()

In [ ]:
#!pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor
models = {
    "Random Forest" : RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=1,
        random_state=42,
        max_depth=10
  ),
    "CatBoostRegressor": CatBoostRegressor(
        iterations=1000,
        learning_rate = 0.01,
        verbose=0
    )
}
maes = {'Random Forest':[], 'CatBoostRegressor':[]}
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for model_name, model in models.items():
  for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    model.fit(X_train, y_train)

    # Validate
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)

    # Store results
    maes[model_name].append(mae)
  print(f"Mean Absolute Error of {model}: {np.mean(maes[model_name])}")

print(f"Average MAE ensemble: {np.mean(list(maes.values()))}")

In [ ]:
np.mean(list(maes.values()))